In [3]:
import os

# Configure Java environment for PySpark
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

print("✓ Environment configured for Spark")

✓ Environment configured for Spark


# Análise — NY Yellow Taxi 2025 (Jan–Mai)

Responde as duas questões do case utilizando os datasets gold gerados pelo pipeline.

In [4]:
from pyspark.sql import functions as F
from ny_rides.shared.spark import get_spark_session

spark = get_spark_session("Questions")
print("Spark session ready")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/15 12:59:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark session ready


## Questão 1 — Qual a média de valor total (total_amount) recebido em um mês?

Considerando todos os yellow táxis da frota.

In [13]:
df_monthly_avg = (
    spark.read.parquet("../data/gold/monthly_average_total_amount")
    .filter((F.col("pickup_year") == 2025) & (F.col("pickup_month") <= 5))
    .orderBy("pickup_year", "pickup_month")
)
df_monthly_avg.show()

+-----------+------------+----------------+
|pickup_year|pickup_month|avg_total_amount|
+-----------+------------+----------------+
|       2025|           1|           25.61|
|       2025|           2|           25.03|
|       2025|           3|           26.27|
|       2025|           4|           26.59|
|       2025|           5|           26.88|
+-----------+------------+----------------+



## Questão 2 — Qual a média de passageiros (passenger_count) por cada hora do dia?

Considerando todos os táxis da frota que pegaram táxi no mês de maio.

In [ ]:
# Read silver data and filter for May (month = 5)
df_may = spark.read.parquet("../data/silver").filter(F.month(F.col("tpep_pickup_datetime")) == 5)

# Calculate average passenger count by hour
df_hourly_avg_may = df_may.groupBy(
    F.hour(F.col("tpep_pickup_datetime")).alias("pickup_hour")
).agg(
    F.avg("passenger_count").alias("avg_passenger_count")
).orderBy("pickup_hour").withColumn(
    "avg_passenger_count",
    F.round(F.col("avg_passenger_count"), 2)
)

df_hourly_avg_may.show(24)

+-----------+-------------------+
|pickup_hour|avg_passenger_count|
+-----------+-------------------+
|          0| 1.3440823142810967|
|          1| 1.3543619683069223|
|          2|  1.380004788634973|
|          3| 1.3670330588874493|
|          4| 1.3594852122262095|
|          5| 1.2155026861089793|
|          6| 1.1668528604012476|
|          7| 1.1831314072693384|
|          8| 1.1876356764027345|
|          9|  1.203529181667379|
|         10|  1.237569343907345|
|         11|  1.257521939610293|
|         12| 1.2731181628165802|
|         13| 1.2830766483170148|
|         14|  1.287297424971319|
|         15| 1.3043874749918578|
|         16| 1.3094486702271133|
|         17| 1.3023525631127117|
|         18|  1.310857096201343|
|         19| 1.3193855956517946|
|         20| 1.3327086520052271|
|         21|  1.345251447452181|
|         22| 1.3542647972167108|
|         23|  1.349555214279505|
+-----------+-------------------+

